# 💰 Business Impact Analysis
Translating model performance into ₪ business value.

In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.loader import load_raw_data
from src.data.preprocessor import clean_data, get_features_and_target
from src.features.engineering import engineer_features

df = clean_data(load_raw_data())
df = engineer_features(df)
X, y = get_features_and_target(df)
model = joblib.load('models/xgb_v1.pkl')
probs = model.predict_proba(X)[:,1]
print(f'Total customers scored: {len(probs):,}')

In [ ]:
# Business parameters
ARR_PER_CUSTOMER = 8000
RETENTION_COST = 500
RETENTION_SUCCESS_RATE = 0.4
THRESHOLD = 0.5

high_risk = (probs >= THRESHOLD).sum()
revenue_at_risk = high_risk * ARR_PER_CUSTOMER
cost_to_intervene = high_risk * RETENTION_COST
saved_customers = int(high_risk * RETENTION_SUCCESS_RATE)
saved_revenue = saved_customers * ARR_PER_CUSTOMER
net_benefit = saved_revenue - cost_to_intervene

print(f'High risk customers: {high_risk:,}')
print(f'Revenue at risk: ₪{revenue_at_risk:,}')
print(f'Cost to intervene: ₪{cost_to_intervene:,}')
print(f'Expected saved customers: {saved_customers:,}')
print(f'Revenue saved: ₪{saved_revenue:,}')
print(f'Net benefit: ₪{net_benefit:,}')
print(f'ROI multiple: {saved_revenue/max(cost_to_intervene,1):.1f}x')

In [ ]:
# Revenue at risk by contract type
df_scored = load_raw_data().iloc[:len(probs)].copy()
df_scored = df_scored.dropna(subset=['TotalCharges']).reset_index(drop=True)
df_scored['churn_probability'] = probs
df_scored['revenue_at_risk'] = (probs * ARR_PER_CUSTOMER).round(0)
contract_risk = df_scored.groupby('Contract')['revenue_at_risk'].sum().sort_values(ascending=False)
print('Revenue at risk by contract type:')
for contract, risk in contract_risk.items():
    print(f'  {contract}: ₪{risk:,.0f}')

In [ ]:
# Sensitivity analysis — ROI at different thresholds
thresholds = np.arange(0.3, 0.9, 0.05)
rois = []
for t in thresholds:
    hr = (probs >= t).sum()
    sc = int(hr * RETENTION_SUCCESS_RATE)
    sr = sc * ARR_PER_CUSTOMER
    ci = hr * RETENTION_COST
    rois.append({'threshold': round(t,2), 'high_risk': hr, 'net_benefit': sr - ci, 'roi_multiple': round(sr/max(ci,1),1)})
roi_df = pd.DataFrame(rois)
print(roi_df.to_string(index=False))

In [ ]:
# Final summary
print('='*50)
print('CHURNGUARD — BUSINESS IMPACT SUMMARY')
print('='*50)
print(f'Model: XGBoost (tuned)')
print(f'AUC: 0.837')
print(f'Recall: 79.1%')
print(f'Customers scored: {len(probs):,}')
print(f'High risk identified: {high_risk:,}')
print(f'Annual revenue saved: ₪{saved_revenue:,}')
print(f'Net annual benefit: ₪{net_benefit:,}')
print(f'ROI: {saved_revenue/max(cost_to_intervene,1):.1f}x')